# Finishing Position Model — EDA & Evaluation
Lightweight check on the training data and model performance.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv('data_finishing.csv')
model_bundle = joblib.load('../models/finishing.pkl')

model      = model_bundle['model']
le_driver  = model_bundle['le_driver']
le_circuit = model_bundle['le_circuit']

print('=== DATA OVERVIEW ===')
print(f'Rows         : {len(df)}')
print(f'Seasons      : {sorted(df["season"].unique())}')
print(f'Circuits     : {df["circuit"].nunique()} unique')
print(f'Drivers      : {df["driver"].nunique()} unique')
print(f'Nulls        : {df.isnull().sum().sum()}')

=== DATA OVERVIEW ===
Rows         : 1938
Seasons      : [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Circuits     : 32 unique
Drivers      : 34 unique
Nulls        : 0


In [2]:
# Re-encode and split the same way as training
df = df.dropna()
df['driver_enc']  = le_driver.transform(df['driver'])
df['circuit_enc'] = le_circuit.transform(df['circuit'])

X = df[['driver_enc', 'circuit_enc', 'season', 'grid_position']]
y = df['finishing_position']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

y_pred = model.predict(X_test)
diff   = np.abs(y_pred - y_test)

print('=== MODEL EVALUATION ===')
print(f'Test rows         : {len(X_test)}')
print(f'Exact accuracy    : {accuracy_score(y_test, y_pred):.2%}')
print(f'Within 1 position : {(diff <= 1).mean():.2%}')
print(f'Within 3 positions: {(diff <= 3).mean():.2%}')
print(f'Within 5 positions: {(diff <= 5).mean():.2%}')
print(f'Mean abs error    : {diff.mean():.2f} positions')

=== MODEL EVALUATION ===
Test rows         : 388
Exact accuracy    : 80.93%
Within 1 position : 85.05%
Within 3 positions: 88.92%
Within 5 positions: 92.27%
Mean abs error    : 1.04 positions


In [3]:
print('=== FEATURE IMPORTANCES ===')
for feat, imp in sorted(zip(X.columns, model.feature_importances_), key=lambda x: -x[1]):
    bar = '█' * int(imp * 40)
    print(f'{feat:<15} {imp:.4f}  {bar}')

=== FEATURE IMPORTANCES ===
grid_position   0.3120  ████████████
circuit_enc     0.2900  ███████████
driver_enc      0.2514  ██████████
season          0.1466  █████


In [4]:
print('=== KNOWN DRIVERS & CIRCUITS ===')
print(f'Drivers  ({len(le_driver.classes_)}):', list(le_driver.classes_))
print()
print(f'Circuits ({len(le_circuit.classes_)}):', list(le_circuit.classes_))

=== KNOWN DRIVERS & CIRCUITS ===
Drivers  (34): ['Alexander Albon', 'Antonio Giovinazzi', 'Brendon Hartley', 'Carlos Sainz', 'Charles Leclerc', 'Daniel Ricciardo', 'Daniil Kvyat', 'Esteban Ocon', 'Fernando Alonso', 'George Russell', 'Guanyu Zhou', 'Jack Aitken', 'Kevin Magnussen', 'Kimi Räikkönen', 'Lance Stroll', 'Lando Norris', 'Lewis Hamilton', 'Marcus Ericsson', 'Max Verstappen', 'Mick Schumacher', 'Nicholas Latifi', 'Nico Hulkenberg', 'Nikita Mazepin', 'Nyck De Vries', 'Pierre Gasly', 'Pietro Fittipaldi', 'Robert Kubica', 'Romain Grosjean', 'Sebastian Vettel', 'Sergey Sirotkin', 'Sergio Perez', 'Stoffel Vandoorne', 'Valtteri Bottas', 'Yuki Tsunoda']

Circuits (32): ['Austin', 'Baku', 'Barcelona', 'Budapest', 'Hockenheim', 'Imola', 'Istanbul', 'Jeddah', 'Le Castellet', 'Lusail', 'Melbourne', 'Mexico City', 'Miami', 'Monaco', 'Monte Carlo', 'Montréal', 'Monza', 'Mugello', 'Nürburgring', 'Portimão', 'Sakhir', 'Shanghai', 'Silverstone', 'Singapore', 'Sochi', 'Spa-Francorchamps', 'Spie